<a href="https://colab.research.google.com/github/elysewong/crmlsanalysis/blob/main/Variablescleaned_Bella.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import zipfile
import pandas as pd

# Correct path to uploaded file
zip_path = "/crmls_last_6_months2.csv.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    # Filter for actual CSV file (not the __MACOSX metadata)
    csv_file = [f for f in z.namelist() if f.endswith('.csv') and not f.startswith('__MACOSX/')][0]

    # Try reading the CSV with latin1 encoding
    with z.open(csv_file) as f:
        df = pd.read_csv(f, encoding='latin1')

# Display the first few rows
df.head()


,ViewYN,BasementYN,PoolPrivateYN,Latitude,Longitude,LivingArea,FireplacesTotal,CountyOrParish,AttachedGarageYN,ParkingTotal,...,BedroomsTotal,StateOrProvince,FireplaceYN,Stories,Levels,LotSizeArea,NewConstructionYN,HighSchoolDistrict,PostalCode,LotSizeSquareFeet
0,False,NaN,NaN,37.330858,-121.849610,1151.0,NaN,Santa Clara,True,0.0,...,8.0,CA,True,NaN,NaN,6418.0,False,Other,95122,6418.0
1,True,NaN,False,34.180411,-118.342020,1434.0,NaN,Los Angeles,False,1.0,...,3.0,CA,True,1.0,One,6473.0,False,Burbank Unified,91505,6473.0
2,False,NaN,False,32.574359,-117.023836,2872.0,NaN,San Diego,True,6.0,...,5.0,CA,True,2.0,Two,5219.0,False,NaN,92154,5219.0
3,True,True,False,37.116859,-122.113773,800.0,NaN,Santa Cruz,False,8.0,...,2.0,CA,False,NaN,NaN,61649.0,False,Other,95006,61649.0
4,True,NaN,False,33.725080,-117.222302,2824.0,NaN,Riverside,True,2.0,...,5.0,CA,False,2.0,Two,7000.0,True,Mendocino Unified,92586,7000.0


In [18]:
# --- 1. Clean Boolean Columns ---
bool_cols = ['ViewYN', 'PoolPrivateYN', 'NewConstructionYN', 'AttachedGarageYN', 'FireplaceYN']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower().map({'true': True, 'false': False})
        df[col] = df[col].fillna(False)  # assuming missing = False

/tmp/ipython-input-18-1479347219.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)  # assuming missing = False
/tmp/ipython-input-18-1479347219.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)  # assuming missing = False
/tmp/ipython-input-18-1479347219.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('futur

In [19]:
# --- 2. Clean Lot Size Columns ---
# Combine all lot size variables into a single 'LotSizeAcres' column
if 'LotSizeAcres' not in df.columns:
    df['LotSizeAcres'] = np.nan

if 'LotSizeSquareFeet' in df.columns:
    df['LotSizeAcres'] = df['LotSizeAcres'].fillna(df['LotSizeSquareFeet'] / 43560)

if 'LotSizeArea' in df.columns:
    df['LotSizeAcres'] = df['LotSizeAcres'].fillna(df['LotSizeArea'])

In [20]:
# Remove large lots > 5 acres
df = df[df['LotSizeAcres'] < 5]

In [21]:
# --- 3. Clean YearBuilt and Compute Building Age ---
current_year = datetime.now().year
if 'YearBuilt' in df.columns:
    df['BuildingAge'] = current_year - df['YearBuilt']
    df.loc[df['BuildingAge'] < 0, 'BuildingAge'] = np.nan  # Fix bad data
    df = df.drop(columns=['YearBuilt'])

In [22]:
# --- 4. Clean Location Columns ---
location_cols = ['CountyOrParish', 'City', 'PostalCode', 'Latitude', 'Longitude', 'HighSchoolDistrict']
for col in location_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(['nan', 'none', ''], np.nan)

In [23]:
# --- 5. Clean Stories/Levels ---
story_cols = ['Stories', 'Levels']
for col in story_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [24]:
# Optionally keep only one
if 'Stories' in df.columns and 'Levels' in df.columns:
    df['StoriesFinal'] = df['Stories'].fillna(df['Levels'])
    df = df.drop(columns=['Stories', 'Levels'])
elif 'Stories' in df.columns:
    df['StoriesFinal'] = df['Stories']
    df = df.drop(columns=['Stories'])
elif 'Levels' in df.columns:
    df['StoriesFinal'] = df['Levels']
    df = df.drop(columns=['Levels'])

In [25]:
# --- 6. Other Core Numeric Variables ---
numeric_cols = ['LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'ParkingTotal']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [26]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 59875 entries, 0 to 61863
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ViewYN                 59875 non-null  bool   
 1   BasementYN             1436 non-null   object 
 2   PoolPrivateYN          59875 non-null  bool   
 3   Latitude               59875 non-null  object 
 4   Longitude              59875 non-null  object 
 5   LivingArea             59839 non-null  float64
 6   FireplacesTotal        0 non-null      float64
 7   CountyOrParish         59875 non-null  object 
 8   AttachedGarageYN       59875 non-null  bool   
 9   ParkingTotal           59875 non-null  float64
 10  LotSizeAcres           59875 non-null  float64
 11  StreetNumberNumeric    59809 non-null  float64
 12  BathroomsTotalInteger  59867 non-null  float64
 13  City                   59831 non-null  object 
 14  BedroomsTotal          59875 non-null  float64
 15  StateOr